# Brazil adaptation sample data: refresh and city ordering

Two jobs, in order:

1. **Refresh** the input tables in `sample/` from the consolidated action bank workbook (Ayinawu, 12 Nov 2026). Only what the workbook holds is touched; every change is listed.
2. **Order** the actions for each test city and write `city-action-order.csv`, the table the city list API reads. Supporting actions (no Direct link to any risk) are not ordered, so they are not in that table.

The ordering uses impact, legal feasibility, cost and policy alignment only. City preferences (timeline, co-benefits, sector, cost) are filters, not scores, as agreed with C40 (*City preferences as filters*, Nov 2026).

Run from `releases/v2/`. Needs `pandas` and `openpyxl`. `sample/` is gitignored: it holds the workbook and the input tables, which are I Care working files.

In [1]:
from pathlib import Path
import re
import openpyxl
import pandas as pd

DB = Path('sample')                                                   # input tables (gitignored)
WORKBOOK = DB / 'Adaptation_Action_Bank_Consolidated.xlsx'            # I Care working file
ORDER = Path('city-action-order.csv')                                 # the one table built here, committed
PREVIOUS_RANKING = Path('../v1/data/output/overall_ranking.csv')       # v1 ranking, for comparison only

# Default ordering: weights as decided, F1 and F3 from the sign-off tracker, alignment from the filters recommendation
WEIGHTS = {'impact': 0.55, 'feasibility': 0.225, 'alignment': 0.225}  # as decided; Impact stays the largest pillar
FEAS_SPLIT = {'legal': 0.5, 'cost': 0.5}                              # F1: equal by default
COST_SCORE = {'low': 0.60, 'medium': 0.50, 'high': 0.40}              # F3: cost band only
# Legal: the 1 to 5 score rescaled to 0 to 1, (score - 1) / 4, as in v1. Alignment: policy alignment score only.

wb = openpyxl.load_workbook(WORKBOOK, read_only=True, data_only=True)

def tab(name, header_row=3, first_row=6):
    """Workbook tab as a DataFrame. Rows are 0-based: the header is on row 3, data from row 6 (rows 4-5 are guidance)."""
    rows = list(wb[name].iter_rows(values_only=True))
    header = [h for h in rows[header_row] if h is not None]
    data = [r[:len(header)] for r in rows[first_row:] if r and r[0]]
    return pd.DataFrame(data, columns=header)

def read(name):
    return pd.read_csv(DB / f'{name}.csv', keep_default_na=False, na_values=[''])

changes = []   # (table, action_id, field, before, after)
def log(table, action_id, field, before, after):
    changes.append({'table': table, 'action_id': action_id, 'field': field, 'before': before, 'after': after})

## 1. Refresh from the workbook

The workbook holds the action bank as I Care and C40 signed it off. Where it and the input tables disagree, the workbook wins, with two exceptions noted below. Finance, policy alignment and city finance are not in the workbook, so their tables are left as they are.

In [2]:
bank = tab('Action').set_index('Action ID')
assert len(bank) == 38

actions = read('adaptation-action').set_index('action_id')
assert set(actions.index) == set(bank.index), 'action IDs differ'

# Names and timeline must already match
for col, wcol in [('action_name_en', 'Action name (EN)'), ('action_name_pt', 'Nome da ação (PT)'), ('timeline', 'Timeline for implementation')]:
    assert (actions[col].str.strip() == bank[wcol].str.strip()).all(), col

# Descriptions: take the workbook text, except one case kept on purpose
KEEP_DB_TEXT = {('ipcc_0085', 'description_en')}   # the workbook text adds "like the Great Barrier Reef"; the earlier text (V7) drops it
for col, wcol in [('description_en', 'Description (EN)'), ('description_pt', 'Descrição (PT)')]:
    for i in actions.index:
        before, after = actions.at[i, col], bank.at[i, wcol]
        if str(before).strip() != str(after).strip() and (i, col) not in KEEP_DB_TEXT:
            log('adaptation-action', i, col, before, after)
            actions.at[i, col] = after

# new0001 and new0002 have no Source in the workbook (OEF to assign). They keep 'icare' until that is decided.
actions.reset_index().to_csv(DB / 'adaptation-action.csv', index=False)

In [3]:
legal = read('actions-legal-feasibility').set_index('action_id')
for i in legal.index:
    score, grade = float(bank.at[i, 'Legal feasibility score']), bank.at[i, 'Legal feasibility grade']
    if legal.at[i, 'legal_score'] != score:
        log('actions-legal-feasibility', i, 'legal_score', legal.at[i, 'legal_score'], score)
    if legal.at[i, 'legal_grade'] != grade:
        log('actions-legal-feasibility', i, 'legal_grade', legal.at[i, 'legal_grade'], grade)
    legal.at[i, 'legal_score'], legal.at[i, 'legal_grade'] = score, grade
legal.reset_index().to_csv(DB / 'actions-legal-feasibility.csv', index=False)

# The norms behind each score are unchanged (same 230 action x norm rows, same authority and competence)
norms_wb = tab('Legal norms')
norms_wb = norms_wb[norms_wb['Norm ID'].astype(str).str.match(r'^nf\d{4}$')]
norms = read('actions-legal-norms')
m = norms.merge(norms_wb, left_on=['action_id', 'norm_id'], right_on=['Action ID', 'Norm ID'], how='outer', indicator=True)
assert (m._merge == 'both').all() and (m.authority == m['Norm authority']).all() and (m.competence == m['Norm competence']).all()
print(f'{len(norms)} legal norm rows unchanged')

230 legal norm rows unchanged


In [4]:
cob_wb = tab('Co-benefits')
cob_wb['final'] = pd.to_numeric(cob_wb['Final score (C40 where given, else I Care)'].astype(str).str.replace('+', '', regex=False))
cob_wb = cob_wb.set_index(['Action ID', 'Co-benefit category'])

cob = read('actions-cobenefits')
assert len(cob) == len(cob_wb) == 304
for k, r in cob.iterrows():
    w = cob_wb.loc[(r.action_id, r.co_benefit)]
    if r.score != w.final:
        log('actions-cobenefits', r.action_id, f'score: {r.co_benefit}', r.score, w.final)
    cob.at[k, 'score'] = int(w.final)
    cob.at[k, 'review_status'] = w['Review status']
    cob.at[k, 'reviewer_notes'] = w['Reviewer notes'] or None
cob.to_csv(DB / 'actions-cobenefits.csv', index=False)

In [5]:
# Pre-actions and co-actions: not in db before. Shown in the UI, never scored.
rel = tab('Action relations').rename(columns={'Action ID': 'action_id', 'Related action ID': 'related_action_id', 'Relation type': 'relation_type'})
rel['relation_type'] = rel.relation_type.str.lower().str.replace('-', '_')     # co_action, pre_action
assert set(rel.action_id) <= set(bank.index) and set(rel.related_action_id) <= set(bank.index)
rel[['action_id', 'related_action_id', 'relation_type']].to_csv(DB / 'actions-relations.csv', index=False)
print(len(rel), 'relations;', rel.relation_type.value_counts().to_dict())

176 relations; {'co_action': 116, 'pre_action': 60}


In [6]:
# Checks: these tables match the workbook already, so nothing is written
def norm(v):
    v = '' if pd.isna(v) else str(v).strip().lower()
    return re.sub(r' (effectiveness|confidence)$', '', v)

links = tab('Risk links', first_row=4)
links = pd.DataFrame({'action_id': links['Action ID'], 'sector': links.Sector.str.lower(), 'component': links.Component.str.lower(),
                      **{k: links[c].map(norm) for k, c in [('eligibility', 'Eligibility / Directness'), ('effectiveness', 'Effectiveness'),
                                                           ('confidence', 'Confidence'), ('maladaptation', 'Maladaptation signal')]},
                      'score_wb': pd.to_numeric(links['Impact scoring'], errors='coerce')})
impact = read('actions-impact')
mine = impact.assign(sector=impact.sector.str.lower(), component=impact.component.str.lower(),
                     **{k: impact[k].map(norm) for k in ['eligibility', 'effectiveness', 'confidence', 'maladaptation']})
m = mine.merge(links, on=['action_id', 'sector', 'component'], suffixes=('', '_wb'))
assert len(m) == 456
for k in ['eligibility', 'effectiveness', 'confidence', 'maladaptation']:
    assert (m[k] == m[f'{k}_wb']).all(), k
assert ((m.component_score.fillna(-1) - m.score_wb.fillna(-1)).abs() < 1e-4).all()

cost = read('actions-financial-cost').set_index('action_id')
assert (cost.cost_band == bank['Cost band'].str.lower().reindex(cost.index)).all()

risk = read('city-risk')
risk_wb = tab('City risk indices', first_row=4)
mr = risk.merge(risk_wb, left_on=['city', 'sector', 'risk'], right_on=['City', 'Sector', 'Risk cell'])
assert len(mr) == 60 and (mr.overall == pd.to_numeric(mr['Overall risk index'])).all()
print('Impact (456 links), cost bands (38) and city risk (60) match the workbook')

Impact (456 links), cost bands (38) and city risk (60) match the workbook


In [7]:
changes = pd.DataFrame(changes)
print('Changes by table:', changes.table.value_counts().to_dict())
grades = changes[changes.field == 'legal_grade']
print(f"\nLegal: {(changes.field == 'legal_score').sum()} scores changed, {len(grades)} grades changed")
grades.assign(score=grades.action_id.map(legal.legal_score))[['action_id', 'before', 'after', 'score']]

Changes by table: {'actions-legal-feasibility': 32, 'actions-cobenefits': 14, 'adaptation-action': 2}

Legal: 23 scores changed, 9 grades changed


,action_id,before,after,score
3,c40_0046,"Feasible, with institutional dependencies","Feasible, with institutional limitations",4.0
5,c40_0048,"Feasible, with institutional limitations",Highly feasible,4.3
11,icare_0171,"Feasible, with institutional dependencies","Feasible, with institutional limitations",3.7
14,icare_0184,"Feasible, with institutional limitations",Highly feasible,4.2
20,ipcc_0090,"Feasible, with institutional dependencies","Feasible, with institutional limitations",3.4
23,ipcc_0094,"Feasible, with institutional limitations",Highly feasible,4.2
29,ipcc_0101,"Feasible, with institutional dependencies","Feasible, with institutional limitations",3.6
30,ipcc_0103,"Feasible, with institutional limitations",Highly feasible,4.2
33,new0002,"Feasible, with institutional limitations",Highly feasible,4.2


In [8]:
changes[changes.table.isin(['adaptation-action', 'actions-cobenefits'])][['table', 'action_id', 'field', 'before', 'after']]

,table,action_id,field,before,after
0,adaptation-action,ipcc_0085,description_pt,NaN,"Reduzir a poluição, a eutrofização e as pressõ..."
1,adaptation-action,new0002,description_pt,Fortalecer o acesso à coleta seletiva de resíd...,Fortalecer o acesso à coleta seletiva de resíd...
34,actions-cobenefits,c40_0046,score: Public health,0,1
35,actions-cobenefits,c40_0046,"score: Biodiversity, habitats and ecosystem se...",0,1
36,actions-cobenefits,c40_0048,score: Social equity and community capacity,0,1
37,actions-cobenefits,c40_0048,"score: Biodiversity, habitats and ecosystem se...",0,1
38,actions-cobenefits,icare_0162,score: Social equity and community capacity,0,1
39,actions-cobenefits,ipcc_0087,score: Social equity and community capacity,0,1
40,actions-cobenefits,ipcc_0091,score: Mobility and access to opportunities an...,0,1
41,actions-cobenefits,ipcc_0094,score: Water quality,0,1


## 2. Order the actions for each city

**Impact** follows the signed-off method (I1 to I5): for each risk cell, the action's raw impact in that sector is weighted by the city's overall risk and by the city's vulnerability and exposure, in proportion to how strongly the action acts on each. The action keeps its best risk cell, and scores are divided by the city's top score. This is checked against I Care's results in the workbook.

**Default ordering** = 0.55 × impact + 0.225 × feasibility + 0.225 × alignment, where feasibility = half legal (rescaled to 0 to 1) and half cost (low 0.6, medium 0.5, high 0.4), and alignment = policy alignment score. Only impact differs between cities.

Every action with a Direct link to a risk the city has gets a position (1, 2, 3 …; equal scores share a position) and an overall score. An action with a Direct link, but not to any risk scored for this city, has neither and is listed last (none in the test cities).

The 9 supporting actions (no Direct link: planning, monitoring and enabling actions) are not ordered. The API shows them inside the detail of the actions they are pre- or co-actions of.

In [9]:
direct = impact[impact.eligibility == 'Direct']
by_sector = (direct.pivot_table(index=['action_id', 'sector'], columns='component', values='component_score', aggfunc='first')
             .reindex(columns=['Vulnerability', 'Exposure']).fillna(0).reset_index())
by_sector.columns = ['action_id', 'sector', 'sv', 'se']
by_sector['raw_impact'] = 1 - (1 - by_sector.sv) * (1 - by_sector.se)

cells = risk.dropna(subset=['overall', 'exposure', 'vulnerability'])          # energy availability has no breakdown, so it is left out (I1)
x = cells.assign(key=cells.sector.str.lower()).merge(by_sector.assign(key=by_sector.sector.str.lower()).drop(columns='sector'), on='key')
x['fit'] = (x.vulnerability * x.sv + x.exposure * x.se) / (x.sv + x.se)
x['potential'] = x.raw_impact * x.overall * x.fit
best = x.loc[x.groupby(['locode', 'action_id']).potential.idxmax(), ['locode', 'city', 'action_id', 'sector', 'risk', 'potential']]
best['impact'] = best.potential / best.groupby('locode').potential.transform('max')

# Check against I Care's city results in the workbook
check = tab('Impact – by city detail', first_row=4)
check = (check.assign(score=pd.to_numeric(check['Normalized impact pillar score']))
         .groupby(['City', 'Action ID']).score.max().rename('impact_wb').reset_index())
c = best.merge(check, left_on=['city', 'action_id'], right_on=['City', 'Action ID'], how='left')
assert (c.impact - c.impact_wb).abs().max() < 1e-3, 'impact differs from the workbook'
print(f'{len(c)} city x action impact scores match the workbook')

145 city x action impact scores match the workbook


In [10]:
cities = risk[['locode', 'city']].drop_duplicates()
grid = cities.merge(actions.reset_index()[['action_id']], how='cross')
grid = grid.merge(best[['locode', 'action_id', 'sector', 'risk', 'potential', 'impact']], how='left')

has_direct = set(direct.action_id)
grid['legal_score'] = grid.action_id.map(legal.legal_score)
grid['legal'] = (grid.legal_score - 1) / 4
grid['cost'] = grid.action_id.map(cost.cost_band).map(COST_SCORE)
grid['feasibility'] = FEAS_SPLIT['legal'] * grid.legal + FEAS_SPLIT['cost'] * grid.cost
grid['alignment'] = grid.action_id.map(read('actions-policy-alignment').set_index('action_id').score)
grid['overall'] = WEIGHTS['impact'] * grid.impact.fillna(0) + WEIGHTS['feasibility'] * grid.feasibility + WEIGHTS['alignment'] * grid.alignment

grid = grid[grid.action_id.isin(has_direct)].copy()   # supporting actions are not ordered
grid['group'] = 'no_city_risk'
grid.loc[grid.impact > 0, 'group'] = 'ordered'

# Only ordered actions get an overall score and a position: outside that group an overall score would suggest an order there is none
ordered = grid.group == 'ordered'
grid.loc[~ordered, 'overall'] = None
grid.loc[ordered, 'position'] = grid[ordered].overall.round(9).groupby(grid.locode).rank(method='dense', ascending=False)
grid['position'] = grid.position.astype('Int64')

# List order: ordered actions by position (ties by impact), then no_city_risk by action ID
grid['_g'] = grid.group.map({'ordered': 0, 'no_city_risk': 1})
grid = grid.sort_values(['locode', '_g', 'position', 'impact', 'action_id'], ascending=[True, True, True, False, True]).drop(columns='_g')

out = grid.rename(columns={'sector': 'impact_sector', 'risk': 'impact_risk', 'potential': 'impact_raw', 'impact': 'impact_score',
                           'legal': 'legal_score_scaled', 'cost': 'cost_score', 'feasibility': 'feasibility_score',
                           'alignment': 'alignment_score', 'overall': 'overall_score'})
COLS = ['locode', 'city', 'action_id', 'position', 'overall_score', 'impact_score', 'impact_raw', 'impact_sector', 'impact_risk',
        'feasibility_score', 'legal_score_scaled', 'cost_score', 'alignment_score']
out[COLS].round(6).to_csv(ORDER, index=False)
out.groupby(['city', 'group']).size().unstack()

group,ordered
city,
Caxias do Sul,29
Itaubal,29
Santa Cruz do Arari,29
Sobral,29
São Luís,29


In [11]:
# Top 10 per city
top = out[out.position <= 10].assign(action=lambda d: d.action_id.map(actions.action_name_en))
top.pivot_table(index='position', columns='city', values='action', aggfunc=lambda s: ' / '.join(s))

city,Caxias do Sul,Itaubal,Santa Cruz do Arari,Sobral,São Luís
position,,,,,
1,Improve climate-resilient water and sanitation...,Implement and strengthen multi-hazard early wa...,Implement and strengthen multi-hazard early wa...,Implementation and strengthening of urban and ...,Improve climate-resilient water and sanitation...
2,Strengthen selective waste collection to reduc...,Develop and implement an extreme weather prepa...,Develop and implement an extreme weather prepa...,Relocate and resettle populations living in hi...,Strengthen selective waste collection to reduc...
3,"Strengthen healthcare capacity, continuity and...",Build and maintain shelters for critical clima...,Improve climate-resilient water and sanitation...,Improve climate-resilient water and sanitation...,"Strengthen healthcare capacity, continuity and..."
4,Relocate and resettle populations living in hi...,"Strengthen healthcare capacity, continuity and...",Strengthen selective waste collection to reduc...,Strengthen climate-resilient food storage and ...,Strengthen programs for control and conservati...
5,"Secure land tenure, essential services and res...",Protect and enhance water sources and watersheds,"Strengthen healthcare capacity, continuity and...",Strengthen programs for control and conservati...,Protect and enhance water sources and watersheds
6,Develop and implement an extreme weather prepa...,"Secure land tenure, essential services and res...",Build and maintain shelters for critical clima...,Protect and enhance water sources and watersheds,Relocate and resettle populations living in hi...
7,Implement and strengthen multi-hazard early wa...,Expand urban green-blue networks and corridors,Protect and enhance water sources and watersheds,"Secure land tenure, essential services and res...",Restore and create natural areas through ecosy...
8,Strengthen programs for control and conservati...,"Stabilize, inspect and maintain slopes in land...",Expand urban green-blue networks and corridors,Restore and create natural areas through ecosy...,"Secure land tenure, essential services and res..."
9,Protect and enhance water sources and watersheds,Strengthen coastal protection infrastructure,"Stabilize, inspect and maintain slopes in land...",Strengthen climate resilience in agriculture a...,Implementation and strengthening of urban and ...


### Compared with the v1 ranking (15 Oct)

v1 used the same method and weights with the earlier legal scores, so the differences below come from the legal update only.

In [12]:
if PREVIOUS_RANKING.exists():
    prev = pd.read_csv(PREVIOUS_RANKING)[['locode', 'action_id', 'rank']].rename(columns={'rank': 'v1_rank'})
    cmp = out[out.group == 'ordered'][['city', 'locode', 'action_id', 'position']].merge(prev, on=['locode', 'action_id'])
    cmp['moved'] = cmp.v1_rank - cmp.position
    summary = cmp.groupby('city').apply(lambda g: pd.Series({
        'same top 10': len(set(g[g.position <= 10].action_id) & set(g[g.v1_rank <= 10].action_id)),
        'actions that moved': int((g.moved != 0).sum()),
        'largest move': int(g.moved.abs().max()),
    }), include_groups=False)
    display(summary)
    display(cmp[cmp.moved.abs() >= 2].assign(action=lambda d: d.action_id.map(actions.action_name_en))
            .sort_values(['city', 'position'])[['city', 'action', 'v1_rank', 'position', 'moved']])

,same top 10,actions that moved,largest move
city,,,
Caxias do Sul,10,12,3
Itaubal,9,14,2
Santa Cruz do Arari,9,10,3
Sobral,10,7,3
São Luís,10,10,3


,city,action,v1_rank,position,moved
12,Caxias do Sul,Implement integrated coastal zone management m...,15.0,13,2.0
17,Caxias do Sul,Reduce pressures on ecosystems to build resili...,16.0,18,-2.0
18,Caxias do Sul,Strengthen coastal protection infrastructure,22.0,19,3.0
34,Itaubal,"Secure land tenure, essential services and res...",8.0,6,2.0
35,Itaubal,Expand urban green-blue networks and corridors,5.0,7,-2.0
37,Itaubal,Strengthen coastal protection infrastructure,11.0,9,2.0
48,Itaubal,Restore and create natural areas through ecosy...,22.0,20,2.0
50,Itaubal,Strengthen sustainable food value chains,20.0,22,-2.0
53,Itaubal,Create or Expand climate risk insurance for sm...,27.0,25,2.0
125,Santa Cruz do Arari,Strengthen coastal protection infrastructure,13.0,10,3.0
